## LG Aimers 9기 — TabM 독립 신호 Forward CV (Colab A100)

기존 LightGBM·CatBoost와 구조가 다른 TabM을 검증한다. R 경기만 과거 시즌으로 학습하고 `season`, `game_type`은 입력에서 제외한다. 범주형 ID embedding과 과거 시즌 정답만 사용하는 temporal target encoding을 결합하며, 07b의 F 예측은 한 값도 바꾸지 않는다. 이 노트북은 제출 ZIP을 만들지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

candidates = [
    Path('/content/drive/MyDrive/LG_aimers_9th'),
    Path('/content/drive/MyDrive/LG_Aimers_9th'),
    Path('/content/drive/MyDrive/LG Aimers 9th'),
    Path.cwd(),
    Path.cwd().parent,
]
required_relative = [
    Path('data/train.csv'),
    Path('src/features.py'),
    Path('src/evaluation.py'),
    Path('src/metrics.py'),
    Path('experiments/08_tabm_cv.py'),
    Path('outputs/f_routing_v1/latest_run.json'),
]
PROJECT_ROOT = next(
    (path for path in candidates if all((path / rel).exists() for rel in required_relative)),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '프로젝트 폴더, experiments/08_tabm_cv.py 또는 07b outputs를 찾지 못했습니다.'
    )
PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT:', PROJECT_ROOT)
print('train.csv size (GB):', round((PROJECT_ROOT / 'data/train.csv').stat().st_size / 1e9, 3))

### 1. TabM 설치와 A100 확인

TabM 공식 PyPI 패키지만 추가로 설치한다. 이 실험은 CPU용이 아니며, 런타임 유형에서 A100 GPU를 선택한 뒤 실행한다.

In [ ]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'tabm==0.0.3'],
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError('tabm==0.0.3 설치 실패')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tabm
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다. Colab 런타임을 A100 GPU로 변경하세요.')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| TabM:', tabm.__version__)
print('GPU:', gpu_name, '| memory (GB):', round(gpu_memory_gb, 1))
if 'A100' not in gpu_name.upper():
    print('주의: 실행은 가능하지만 이 노트북의 권장 GPU는 A100입니다.')

### 2. 07b 기준 OOF와 선택 경로 확인

In [ ]:
ROUTING_OUTPUT_ROOT = PROJECT_ROOT / 'outputs/f_routing_v1'
with (ROUTING_OUTPUT_ROOT / 'latest_run.json').open(encoding='utf-8') as handle:
    routing_latest = json.load(handle)
recorded_source = Path(routing_latest['run_dir'])
SOURCE_RUN_DIR = next(
    (path for path in [recorded_source, ROUTING_OUTPUT_ROOT / recorded_source.name] if path.exists()),
    None,
)
if SOURCE_RUN_DIR is None:
    raise FileNotFoundError('07b 최신 run 폴더를 찾지 못했습니다.')
with (SOURCE_RUN_DIR / 'summary.json').open(encoding='utf-8') as handle:
    routing_summary = json.load(handle)
source_oof = SOURCE_RUN_DIR / 'route_oof.csv.gz'
if not source_oof.exists():
    raise FileNotFoundError(source_oof)
display(pd.DataFrame([
    ['07b signature', routing_summary['signature']],
    ['selected route', routing_summary['selected_route']],
    ['selection status', routing_summary['selection_status']],
    ['recent Brier', routing_summary['selected_recent_brier']],
    ['recent BSS raw', routing_summary['selected_recent_bss_raw']],
    ['route OOF size (MB)', round(source_oof.stat().st_size / 1e6, 1)],
], columns=['check', 'value']))
print('SOURCE_RUN_DIR:', SOURCE_RUN_DIR)

### 3. 실험 가설과 고정 조건

점수를 본 뒤 조건을 바꾸지 않도록 아래 후보와 안전 기준을 먼저 고정한다. BCE와 Brier loss는 동일한 TabM 구조·seed로 비교하고, 기존 예측과의 단순 확률 평균 비중은 10·20·30·40%만 검사한다.

In [ ]:
experiment_contract = pd.DataFrame([
    ['학습 행', 'valid season보다 과거인 R 경기만'],
    ['검증 행', '2022·2023·2024의 R/F 전체'],
    ['입력 제외', 'season, game_type'],
    ['범주형 처리', 'fold train-only vocabulary + embedding, unknown=0'],
    ['target encoding', '각 학습 행보다 엄격히 과거 시즌 정답만 사용'],
    ['loss', 'BCE와 Brier 각각, k개 예측별 loss 후 평균'],
    ['추론', 'k개 logit이 아니라 k개 probability 평균'],
    ['혼합 후보', 'TabM 10%, 20%, 30%, 40%'],
    ['F 처리', '07b 기준 예측과 bit-for-bit 동일'],
    ['안전 기준', '최근 R Brier 1e-5 이상 개선 + fold 최대 악화 2e-5 이하'],
], columns=['item', 'fixed rule'])
display(experiment_contract)

### 4. 실행 설정

A100 기준 기본 설정이다. 이미 완료된 fold/loss 캐시는 자동 재사용한다. 설정이나 코드가 바뀌면 signature가 달라져 새 run 폴더가 생긴다.

In [ ]:
VALID_SEASONS = [2022, 2023, 2024]
LOSSES = ['bce', 'brier']
BATCH_SIZE = 4096
MAX_EPOCHS = 15
PATIENCE = 3
HALF_LIFE = 1.5
K = 32
N_BLOCKS = 3
D_BLOCK = 256
DROPOUT = 0.10
NUM_WORKERS = 0
FORCE_RERUN = False
OUTPUT_ROOT = PROJECT_ROOT / 'outputs/tabm_cv_v1'

settings = pd.DataFrame([
    ['valid seasons', VALID_SEASONS],
    ['losses', LOSSES],
    ['batch size', BATCH_SIZE],
    ['max epochs / patience', f'{MAX_EPOCHS} / {PATIENCE}'],
    ['recency half-life', HALF_LIFE],
    ['TabM k', K],
    ['blocks / width / dropout', f'{N_BLOCKS} / {D_BLOCK} / {DROPOUT}'],
    ['force rerun', FORCE_RERUN],
], columns=['setting', 'value'])
display(settings)

### 5. TabM Forward CV 실행

총 3개 fold × 2개 loss를 학습한다. 각 epoch에서 R validation Brier를 기준으로 조기 종료하며, 중간에 런타임이 끊겨도 완료된 fold/loss 캐시부터 이어서 실행한다.

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments/08_tabm_cv.py'),
    '--train-path', str(PROJECT_ROOT / 'data/train.csv'),
    '--routing-output-root', str(ROUTING_OUTPUT_ROOT),
    '--source-run-dir', str(SOURCE_RUN_DIR),
    '--output-dir', str(OUTPUT_ROOT),
    '--valid-seasons', *map(str, VALID_SEASONS),
    '--losses', *LOSSES,
    '--device', 'cuda',
    '--batch-size', str(BATCH_SIZE),
    '--max-epochs', str(MAX_EPOCHS),
    '--patience', str(PATIENCE),
    '--half-life', str(HALF_LIFE),
    '--k', str(K),
    '--n-blocks', str(N_BLOCKS),
    '--d-block', str(D_BLOCK),
    '--dropout', str(DROPOUT),
    '--num-workers', str(NUM_WORKERS),
]
if FORCE_RERUN:
    command.append('--force-rerun')
print('Running:', ' '.join(command))
started = time.perf_counter()
subprocess.run(command, check=True)
print('elapsed minutes:', round((time.perf_counter() - started) / 60, 1))

### 6. 최신 run과 무결성 확인

In [ ]:
with (OUTPUT_ROOT / 'latest_run.json').open(encoding='utf-8') as handle:
    latest = json.load(handle)
recorded_run = Path(latest['run_dir'])
RUN_DIR = next(
    (path for path in [recorded_run, OUTPUT_ROOT / recorded_run.name] if path.exists()),
    None,
)
if RUN_DIR is None:
    raise FileNotFoundError('08 최신 run 폴더를 찾지 못했습니다.')
summary = latest['summary']
audit = summary['integrity_audit']
display(pd.DataFrame([
    ['signature', summary['signature']],
    ['rows', audit['rows']],
    ['training scope', audit['training_scope']],
    ['season excluded', audit['season_feature_excluded_all_folds']],
    ['game_type excluded', audit['game_type_feature_excluded_all_folds']],
    ['temporal TE audit', audit['temporal_target_encoding_passed_all_folds']],
    ['F exact', audit['F_predictions_exact']],
    ['selected', summary['selected_candidate']],
    ['status', summary['selection_status']],
], columns=['check', 'value']))
if audit['status'] != 'passed' or not audit['F_predictions_exact']:
    raise RuntimeError('08 무결성 검사를 통과하지 못했습니다.')
print('RUN_DIR:', RUN_DIR)

### 7. 학습 과정과 조기 종료 확인

In [ ]:
fold_training = pd.read_csv(RUN_DIR / 'fold_training.csv')
history = pd.read_csv(RUN_DIR / 'training_history.csv')
display(fold_training.sort_values(['valid_season', 'loss']).reset_index(drop=True))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for axis, season in zip(axes, VALID_SEASONS):
    subset = history[history['valid_season'].eq(season)]
    for loss_name, group in subset.groupby('loss'):
        axis.plot(group['epoch'], group['valid_r_brier'], marker='o', label=loss_name)
    axis.set_title(str(season))
    axis.set_xlabel('epoch')
    axis.grid(alpha=0.25)
axes[0].set_ylabel('R validation Brier (lower is better)')
axes[-1].legend()
plt.tight_layout()
plt.show()

### 8. TabM 단독 성능

먼저 혼합 없이 독립 모델 자체가 어느 정도의 Brier와 BSS를 내는지 본다. `recent_r_brier`가 핵심이며, 세 fold가 같은 방향인지 함께 확인한다.

In [ ]:
model_summary = pd.read_csv(RUN_DIR / 'model_summary.csv')
display(model_summary[[
    'model', 'pooled_r_brier', 'pooled_r_bss_raw',
    'recent_r_brier', 'recent_r_bss_raw', 'recent_r_roc_auc',
    'r_brier_2022', 'r_brier_2023', 'r_brier_2024',
]])

### 9. 기존 앙상블과 예측 다양성

단독 점수가 조금 낮아도 기존 모델과 오류가 다르면 앙상블에 가치가 있다. 상관계수와 평균 절대 예측 차이를 최근 R 구간 중심으로 확인한다.

In [ ]:
correlations = pd.read_csv(RUN_DIR / 'prediction_correlations.csv')
recent_r_corr = correlations[
    correlations['partition'].eq('recent_pooled') & correlations['scope'].eq('R')
]
display(recent_r_corr[[
    'model', 'pearson_vs_base', 'mean_abs_prediction_delta_vs_base'
]].sort_values('pearson_vs_base'))

### 10. 고정 비중 혼합 결과와 안전 게이트

In [ ]:
blend_summary = pd.read_csv(RUN_DIR / 'blend_summary.csv')
display(blend_summary[[
    'model', 'recent_r_brier', 'recent_r_bss_raw',
    'delta_recent_r_brier_vs_base', 'worst_fold_delta',
    'improves_recent_enough', 'stable_across_folds', 'passes_fixed_gate',
]].head(12))

candidates = blend_summary[~blend_summary['model'].eq('base_07b')].copy()
plt.figure(figsize=(9, 4))
colors = np.where(candidates['passes_fixed_gate'], '#2ca02c', '#9aa0a6')
plt.bar(candidates['model'], -candidates['delta_recent_r_brier_vs_base'], color=colors)
plt.axhline(0, color='black', linewidth=1)
plt.ylabel('recent R Brier improvement vs 07b')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 11. Fold별 안정성과 F 고정 확인

In [ ]:
selected_name = summary['selected_candidate']
selected_row = blend_summary[blend_summary['model'].eq(selected_name)].iloc[0]
fold_check = pd.DataFrame([
    [season, selected_row[f'r_brier_{season}'], selected_row[f'delta_r_brier_{season}_vs_base']]
    for season in VALID_SEASONS
], columns=['valid_season', 'selected R Brier', 'delta vs 07b'])
display(fold_check)
print('selected F max abs difference:', audit['selected_f_prediction_max_abs_difference'])
assert audit['selected_f_prediction_max_abs_difference'] == 0.0

### 12. Temporal target encoding 누수 감사

각 fold의 validation encoding은 train season만, train encoding은 해당 행보다 과거 season만 사용해야 한다. 그룹별 validation 미지 카테고리 비율도 같이 본다.

In [ ]:
with (RUN_DIR / 'target_encoding_audit.json').open(encoding='utf-8') as handle:
    te_audit = json.load(handle)
te_rows = []
for season, fold in te_audit.items():
    assert fold['status'] == 'passed'
    assert fold['training_encodings_use_strictly_earlier_seasons']
    assert fold['validation_uses_strictly_earlier_seasons']
    for group_name, group in fold['groups'].items():
        te_rows.append({
            'valid_season': int(season),
            'group': group_name,
            'known_fraction': group['validation_known_fraction'],
            'effective_count_mean': group['validation_effective_count_mean'],
        })
display(pd.DataFrame(te_rows).pivot(index='group', columns='valid_season', values='known_fraction'))

### 13. 자동 판정과 다음 단계

In [ ]:
decision = pd.DataFrame([
    ['selected candidate', summary['selected_candidate']],
    ['selection status', summary['selection_status']],
    ['07b recent R Brier', summary['base_recent_r_brier']],
    ['selected recent R Brier', summary['selected_recent_r_brier']],
    ['recent R delta', summary['selected_delta_recent_r_brier_vs_base']],
    ['worst fold delta', summary['selected_worst_fold_delta']],
    ['F exact', audit['F_predictions_exact']],
], columns=['item', 'value'])
display(decision)
if summary['selected_candidate'] == 'base_07b':
    print('판정: 현재 TabM 설정은 최종 모델로 승격하지 않는다. 단독/상관/학습 결과로 다음 한 축을 수정한다.')
else:
    print('판정: 안전 게이트를 통과했다. 선택 loss와 비중을 전체 학습 제출 노트북에서 재현한다.')

### 14. 검토용 결과 ZIP 생성

실행 결과가 저장된 `08_tabm_cv.ipynb`와 아래 `tabm_review.zip`을 전달한다. 행별 분석이 필요하면 `oof_predictions.csv.gz`도 함께 전달한다.

In [ ]:
review_names = [
    'summary.json',
    'configuration.json',
    'integrity_audit.json',
    'feature_manifest.json',
    'target_encoding_audit.json',
    'model_summary.csv',
    'blend_summary.csv',
    'prediction_correlations.csv',
    'fold_training.csv',
    'training_history.csv',
]
REVIEW_ZIP = OUTPUT_ROOT / 'tabm_review.zip'
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        path = RUN_DIR / name
        if not path.exists():
            raise FileNotFoundError(path)
        archive.write(path, arcname=name)
OOF_PATH = RUN_DIR / 'oof_predictions.csv.gz'
print('검토 ZIP:', REVIEW_ZIP)
print('검토 ZIP 크기 (MB):', round(REVIEW_ZIP.stat().st_size / 1e6, 2))
print('OOF:', OOF_PATH)
print('OOF 크기 (MB):', round(OOF_PATH.stat().st_size / 1e6, 2))
print('전달: 실행된 08_tabm_cv.ipynb + tabm_review.zip (+ oof_predictions.csv.gz)')